# Inspeção inicial de um snapshot do CNPJ

Este notebook prepara um snapshot mensal a partir do ZIP original e valida se a extração ficou pronta para uso no POC.

## Fluxo recomendado no Colab

1. Manter no Google Drive apenas o arquivo mensal original, por exemplo `2026-04.zip`.
2. Montar o Drive no Colab.
3. Copiar o ZIP do snapshot alvo para `/content`.
4. Executar este notebook para montar a estrutura de trabalho em `/content/dados_brutos/cnpj/AAAA-MM/`.
5. Validar famílias, amostras e metadados.

Este notebook não depende de `subarquivos_zip` persistente. A camada intermediária é criada de forma temporária e removida ao final da preparação.

In [ ]:
from collections import defaultdict
from pathlib import Path
import json
import shutil
import zipfile
from google.colab import drive

SNAPSHOT_MES = '2026-04'
NOME_ARQUIVO_ZIP = '2026-04.zip'
DRIVE_RAIZ = Path('/content/drive/MyDrive/geoBusiness/dados_brutos/cnpj')
CAMINHO_ZIP_DRIVE = DRIVE_RAIZ / SNAPSHOT_MES / 'original' / NOME_ARQUIVO_ZIP
ENTRADA_LOCAL = Path('/content/entrada_cnpj')
CAMINHO_ZIP_PRINCIPAL = ENTRADA_LOCAL / NOME_ARQUIVO_ZIP
RAIZ_DADOS = Path('/content/dados_brutos/cnpj')

drive.mount('/content/drive')
ENTRADA_LOCAL.mkdir(parents=True, exist_ok=True)
if not CAMINHO_ZIP_DRIVE.exists():
    raise FileNotFoundError(f'Arquivo não encontrado no Drive: {CAMINHO_ZIP_DRIVE}')

precisa_recopiar = False

if not CAMINHO_ZIP_PRINCIPAL.exists():
    precisa_recopiar = True
else:
    tamanho_drive = CAMINHO_ZIP_DRIVE.stat().st_size
    tamanho_local = CAMINHO_ZIP_PRINCIPAL.stat().st_size

    if tamanho_local != tamanho_drive:
        print('Arquivo local incompleto ou divergente. Será copiado novamente.')
        CAMINHO_ZIP_PRINCIPAL.unlink()
        precisa_recopiar = True
    elif not zipfile.is_zipfile(CAMINHO_ZIP_PRINCIPAL):
        print('Arquivo local inválido. Será copiado novamente.')
        CAMINHO_ZIP_PRINCIPAL.unlink()
        precisa_recopiar = True

if precisa_recopiar:
    shutil.copy2(CAMINHO_ZIP_DRIVE, CAMINHO_ZIP_PRINCIPAL)

print('ZIP no Drive:', CAMINHO_ZIP_DRIVE)
print('ZIP local:', CAMINHO_ZIP_PRINCIPAL)
print('Tamanho no Drive:', CAMINHO_ZIP_DRIVE.stat().st_size)
print('Tamanho local:', CAMINHO_ZIP_PRINCIPAL.stat().st_size)
print('ZIP local válido?', zipfile.is_zipfile(CAMINHO_ZIP_PRINCIPAL))
CAMINHO_ZIP_PRINCIPAL

In [ ]:
FAMILIAS = {
    'Empresas': 'empresas',
    'Estabelecimentos': 'estabelecimentos',
    'Socios': 'socios',
    'Simples': 'simples',
    'Cnaes': 'dominios',
    'Motivos': 'dominios',
    'Municipios': 'dominios',
    'Naturezas': 'dominios',
    'Paises': 'dominios',
    'Qualificacoes': 'dominios',
}

def classificar_zip(nome: str) -> str:
    for prefixo, familia in FAMILIAS.items():
        if nome.startswith(prefixo):
            return familia
    raise ValueError(f'ZIP interno não reconhecido: {nome}')

def preparar_snapshot(caminho_zip_principal: Path, raiz_dados: Path, snapshot_mes: str) -> Path:
    if not caminho_zip_principal.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho_zip_principal}')

    snapshot = raiz_dados / snapshot_mes
    pacote_original = snapshot / 'original'
    extraido_texto = snapshot / 'extraido'
    processado = snapshot / 'processado'
    metadados = snapshot / 'metadados'
    temporario = snapshot / '_tmp_subarquivos_zip'

    for pasta in [pacote_original, extraido_texto, processado / 'recorte', processado / 'preparado', processado / 'analitico', metadados, temporario]:
        pasta.mkdir(parents=True, exist_ok=True)

    destino_zip = pacote_original / caminho_zip_principal.name
    if caminho_zip_principal.resolve() != destino_zip.resolve() and not destino_zip.exists():
        shutil.copy2(caminho_zip_principal, destino_zip)
    caminho_zip_principal = destino_zip

    with zipfile.ZipFile(caminho_zip_principal) as zip_principal:
        zip_principal.extractall(temporario)

    zips_internos = sorted(
        arquivo for arquivo in temporario.rglob('*.zip') if arquivo.is_file()
    )

    if not zips_internos:
        raise FileNotFoundError(
            f'Nenhum ZIP interno foi encontrado dentro de {caminho_zip_principal}'
        )

    for zip_interno in zips_internos:
        familia = classificar_zip(zip_interno.name)
        destino_familia = extraido_texto / familia
        destino_familia.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_interno) as zip_familia:
            zip_familia.extractall(destino_familia)

    shutil.rmtree(temporario)
    return snapshot

SNAPSHOT = preparar_snapshot(CAMINHO_ZIP_PRINCIPAL, RAIZ_DADOS, SNAPSHOT_MES)
SNAPSHOT

In [ ]:
partes = [
    'original',
    'extraido',
    'processado',
    'metadados',
]

estrutura = {parte: (SNAPSHOT / parte).exists() for parte in partes}
estrutura

In [ ]:
familias = {
    'empresas': SNAPSHOT / 'extraido' / 'empresas',
    'estabelecimentos': SNAPSHOT / 'extraido' / 'estabelecimentos',
    'simples': SNAPSHOT / 'extraido' / 'simples',
    'socios': SNAPSHOT / 'extraido' / 'socios',
    'dominios': SNAPSHOT / 'extraido' / 'dominios',
}

resumo_familias = {}
for familia, pasta in familias.items():
    arquivos = sorted([arquivo for arquivo in pasta.iterdir() if arquivo.is_file()]) if pasta.exists() else []
    resumo_familias[familia] = {
        'quantidade_arquivos': len(arquivos),
        'primeiros_arquivos': [arquivo.name for arquivo in arquivos[:3]],
        'bytes_total': sum(arquivo.stat().st_size for arquivo in arquivos),
    }

resumo_familias

In [ ]:
def amostrar_linhas(caminho: Path, quantidade: int = 3):
    linhas = []
    with caminho.open('r', encoding='latin-1') as arquivo:
        for _ in range(quantidade):
            linha = arquivo.readline()
            if not linha:
                break
            linhas.append(linha.strip())
    return linhas

arquivos_amostra = {
    'cnae': SNAPSHOT / 'extraido' / 'dominios' / 'F.K03200$Z.D60411.CNAECSV',
    'municipios': SNAPSHOT / 'extraido' / 'dominios' / 'F.K03200$Z.D60411.MUNICCSV',
    'empresas': next((SNAPSHOT / 'extraido' / 'empresas').iterdir()),
    'estabelecimentos': next((SNAPSHOT / 'extraido' / 'estabelecimentos').iterdir()),
    'simples': next((SNAPSHOT / 'extraido' / 'simples').iterdir()),
}

{chave: amostrar_linhas(caminho) for chave, caminho in arquivos_amostra.items()}

In [ ]:
manifesto = {
    'snapshot_mes': SNAPSHOT_MES,
    'origem_zip': str((SNAPSHOT / 'original' / '2026-04.zip')),
    'camada_intermediaria_persistida': False,
    'estrutura': estrutura,
    'resumo_familias': resumo_familias,
}

saida = SNAPSHOT / 'metadados' / 'manifest_colab.json'
saida.write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8')
saida